In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

In [2]:
# ==========================================
# 1. KHAI BÁO ĐƯỜNG DẪN
# ==========================================
base_model_name = "Qwen/Qwen2.5-3B"

# SỬA DÒNG NÀY: Trỏ đúng vào thư mục checkpoint của bạn trên Kaggle
adapter_path = "/kaggle/input/models/huymaiquang/model-medqa/transformers/default/1/medqa-qwen-grpo/checkpoint-300" 


In [3]:

# ==========================================
# 2. LOAD TOKENIZER & BASE MODEL
# ==========================================
print("Đang load Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)

print("Đang load Base Model (Sẽ rất nhanh trên Kaggle)...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=torch.bfloat16, 
    device_map="auto",
    trust_remote_code=True
)

Đang load Tokenizer...


config.json:   0%|          | 0.00/683 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Đang load Base Model (Sẽ rất nhanh trên Kaggle)...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [4]:
# ==========================================
# 3. GẮN LORA ADAPTER VÀO MODEL GỐC
# ==========================================
print("Đang đắp bản vá LoRA y khoa vào mô hình...")
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval() 
print("✅ Bác sĩ AI đã sẵn sàng khám bệnh!")


Đang đắp bản vá LoRA y khoa vào mô hình...
✅ Bác sĩ AI đã sẵn sàng khám bệnh!


In [5]:
# ==========================================
# 4. HÀM ĐẶT CÂU HỎI
# ==========================================
PROMPT_TEMPLATE = """A conversation between User and Assistant.
The assistant thinks internally and then answers.

The reasoning must be inside <think></think>
The final answer must be inside <answer></answer>

User:
{question}

Options:
{options}

Assistant:
"""

def test_medical_ai(question, options_dict):
    options_text = "\n".join([f"{k}. {v}" for k, v in options_dict.items()])
    prompt = PROMPT_TEMPLATE.format(question=question, options=options_text)
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    print("\n🩺 Đang phân tích ca bệnh...\n" + "="*50)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response_tokens = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(response_tokens, skip_special_tokens=True)

In [6]:

# ==========================================
# 5. CHẠY THỬ
# ==========================================
cau_hoi = "A 45-year-old woman presents with fatigue, weight gain, and cold intolerance. Lab results show elevated TSH and low free T4. What is the most likely diagnosis?"
dap_an = {
    "A": "Hyperthyroidism",
    "B": "Hypothyroidism",
    "C": "Type 2 Diabetes",
    "D": "Cushing's syndrome"
}

print(test_medical_ai(cau_hoi, dap_an))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



🩺 Đang phân tích ca bệnh...
<think>
The patient's symptoms of fatigue, weight gain, and cold intolerance are consistent with hypothyroidism. Elevated TSH and low free T4 levels further support this diagnosis. Hyperthyroidism would typically present with weight loss, increased appetite, and heat intolerance. Type 2 diabetes is characterized by high blood sugar levels and may present with fatigue, but it does not typically cause cold intolerance. Cushing's syndrome is associated with weight gain, but it does not typically cause fatigue or cold intolerance.
</think>

<answer>
B. Hypothyroidism</answer>
